In [ ]:
import numpy as np
import os
import netCDF4 as nc  
from tqdm import tqdm
import sys
import importlib

Rips(maxdim=1, thresh=inf, coeff=2, do_cocycles=False, n_perm = None, verbose=True)


In [3]:
# Get the path of the notebook file
notebook_path = os.getcwd()

# Directory containing the notebook file
notebook_directory = os.path.dirname(notebook_path)
notebook_directory

'/Users/himanshuyadav/UF Dropbox/Himanshu Yadav/files/PhD_TDA/Climate_edit'

In [4]:
sys.path.append(os.path.join(notebook_directory, 'src'))
import cubical_pers_and_filt_visual, globe_visualization, feature_tracking

importlib.reload(cubical_pers_and_filt_visual)
importlib.reload(globe_visualization)
importlib.reload(feature_tracking)

<module 'feature_tracking' from '/Users/himanshuyadav/UF Dropbox/Himanshu Yadav/files/PhD_TDA/Climate_edit/src/feature_tracking.py'>

# Loading the data

In [5]:
dSet_SL = nc.Dataset(notebook_directory+"/data/raw/slp.daily.nc")
# Extract variables from the dataset
# This example assumes there are time, lat, lon, and a variable 'hgt'
time = dSet_SL.variables['time'][:]
lat = dSet_SL.variables['lat'][:]
lon = dSet_SL.variables['lon'][:]
slp = dSet_SL.variables['slp'][:]

In [6]:
lat_cut = [[0,90]]
lon_cut = [[0,360]]

In [7]:
# Function to check if a value is within any of the specified ranges
def is_in_ranges(value, ranges):
    for r in ranges:
        if r[0] <= value <= r[1]:
            return True
    return False

In [8]:
def give_index(list,list_cut):
    indices = []
    for i in range(len(list.data)):
        if is_in_ranges(list.data[i],list_cut):
            indices.append(i)
    return indices

lon_index = give_index(lon,lon_cut)
lat_index = give_index(lat,lat_cut)

In [9]:
slp_new = slp[:,lat_index,:][:,:,lon_index]

In [10]:
slp.data.shape

(27919, 73, 144)

## Flattening the HGT

In [11]:
slp_list = []
for i in range(len(slp_new)):
    slp_list.append(np.array(np.hstack((slp_new[i][:, 21:], slp_new[i][:, 0:21])).data.flatten()))

# Normalizing the data

In [12]:
start_year = 1948 

In [13]:
def del_29_Feb(reading_list,start_year):
    step = 0
    while start_year + step < 2024:
        if (start_year+step)%4 == 0:
            del reading_list[step*365+59]
        step += 1
    return reading_list[:step*365]

In [14]:
slp_list = del_29_Feb(slp_list,1948)

In [15]:
def av_day(reading_list):
    num_year = int(len(reading_list)/365)
    av_list = reading_list[:365]

    for i in range(1,num_year):
        for j in range(365):
            av_list[j] = av_list[j] + reading_list[i*365+j]
    
    for i in range(len(av_list)):
        av_list[i] = av_list[i]*1/num_year
    
    for i in range(num_year):
        for j in range(365):
            reading_list[i*365+j] = reading_list[i*365+j] - av_list[j]

    return reading_list

In [16]:
slp_list = av_day(slp_list)

# Tracking

#### Saving to load for julia

In [ ]:
# Base directory
base_dir = notebook_directory +"/data/processed_data/SLP_data_years"

for j in tqdm(range(2024-1948)):
    year = 1948 + j  # Calculate the actual year
    slp_cut = slp_list[j*365:(j+1)*365]
    
    # Create year directory if it doesn't exist
    year_dir = os.path.join(base_dir, str(year))
    os.makedirs(year_dir, exist_ok=True)
    
    for i in range(365):
        # Positive persistences (using -X)
        polar_X = globe_visualization.reshape_to_polar_matrix(slp_cut[i], lat_index, lon_index, default_value="smallest")
        
        # Create filename
        filename = f"slp_sup_{year}_day_{i+1}.npy"
        filepath = os.path.join(year_dir, filename)

        # Save the array
        np.save(filepath, -polar_X)

        # Negative persistences (using X)
        polar_X = globe_visualization.reshape_to_polar_matrix(slp_cut[i], lat_index, lon_index, default_value="largest")
        
        # Create filename
        filename = f"slp_sub_{year}_day_{i+1}.npy"
        filepath = os.path.join(year_dir, filename)

        # Save the array
        np.save(filepath, polar_X)
